1. The Problem With Accuracy

Imagine a fraud detection system:

-10,000 transactions
-Only 800 are fraud
-9,200 are normal

A model predicts:

     “Everything is normal.”

Accuracy becomes:

Accuracy=
10000/9200=92%

Looks amazing.

But the model detects:

Fraud caught = 0
Business value = 0

This is why accuracy alone is dangerous.

2. Build the “Useless but Accurate” Model

In [ ]:
import numpy as np
import pandas as pd

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report
)


# Create highly imbalanced dataset


X, y = make_classification(
    n_samples=10000,
    n_features=10,
    weights=[0.92, 0.08],   # 92% class 0, 8% class 1
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# USELESS MODEL


y_pred = np.zeros_like(y_test)


# Metrics


acc = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred)

print("Accuracy :", acc)
print("Precision:", precision)
print("Recall   :", recall)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

3. Expected Output
Accuracy : 0.92
Precision: 0.0
Recall   : 0.0

4. What the Confusion Matrix Reveals
	Predicted Normal	Predicted Fraud
Actual Normal	1840	0
Actual Fraud	160	0

Interpretation:

Model never catches fraud
Accuracy looks excellent
Business loses money continuously

5. Why Precision & Recall Matter
Precision

“When the model says YES… how often is it correct?”

Precision=
TP+FP
TP
	​


Use when:

False alarms are expensive
Example:
Spam detection
Loan approval
Ad targeting
Recall

“Out of all real positives… how many did we catch?”

Recall=TP/TP+FN

Use when:

Missing positives is dangerous
Example:
-Fraud detection
-Cancer detection
-Cybersecurity

6. Real Business Interpretation
| Business Case      | Priority Metric | Why                                  |
| ------------------ | --------------- | ------------------------------------ |
| Fraud Detection    | Recall          | Missing fraud costs money            |
| Medical Diagnosis  | Recall          | Missing disease is dangerous         |
| Spam Filter        | Precision       | Wrongly flagging emails annoys users |
| Loan Approval      | Precision       | Bad approvals cost money             |
| Marketing Campaign | Precision       | Reduce wasted ad spend               |


7. Threshold Tuning (Most Important Part)

Most ML models output probabilities:

In [ ]:
0.12
0.87
0.43
0.95

Default threshold:

In [ ]:
if probability > 0.5:
    predict = 1

But businesses rarely want the default threshold.

8. Example: Fraud Detection
Conservative Threshold
threshold = 0.9

Result:

Very few fraud alerts
High precision
Low recall

Meaning:

“Only alert when highly confident.”

Aggressive Threshold
threshold = 0.2

Result:
Many fraud alerts
Lower precision
Higher recall
Meaning:
“Catch as much fraud as possible.”

9. Threshold Tuning Code

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score

# Train real model
model = LogisticRegression()
model.fit(X_train, y_train)

# Probability predictions
probs = model.predict_proba(X_test)[:, 1]

thresholds = [0.1, 0.3, 0.5, 0.7, 0.9]

print("Threshold | Precision | Recall")

for t in thresholds:
    preds = (probs >= t).astype(int)

    p = precision_score(y_test, preds)
    r = recall_score(y_test, preds)

    print(f"{t:^9} | {p:^9.2f} | {r:^7.2f}")

10. Typical Output

| Threshold | Precision | Recall |
| --------- | --------- | ------ |
| 0.1       | 0.32      | 0.95   |
| 0.3       | 0.58      | 0.82   |
| 0.5       | 0.74      | 0.63   |
| 0.7       | 0.89      | 0.41   |
| 0.9       | 0.97      | 0.12   |


11. Business Insight

Lower Threshold:

More detections
More false alarms
Higher recall

Higher Threshold:

Cleaner predictions
Fewer detections
Higher precision

This is not a technical decision.

It is a business decision.

12. Metrics Cheat Sheet

| Metric    | Formula                             | Best For              | Dangerous When                  |
| --------- | ----------------------------------- | --------------------- | ------------------------------- |
| Accuracy  | Correct / Total                     | Balanced datasets     | Imbalanced data                 |
| Precision | TP / (TP+FP)                        | Avoid false positives | Missing positives matters       |
| Recall    | TP / (TP+FN)                        | Catch all positives   | False alarms costly             |
| F1 Score  | Harmonic mean of precision & recall | Balance both          | Business priorities unclear     |
| ROC-AUC   | Ranking quality                     | Comparing models      | Threshold matters in production |


13. Executive-Level Explanation

“Our model is 92% accurate, but accuracy is misleading because the dataset is imbalanced.
The model fails to identify the minority class entirely.
Precision and recall expose the true performance.
Threshold tuning allows us to optimize predictions based on business risk tolerance.”

14. One-Line Takeaway

A high-accuracy model can still fail the business completely.
Precision and recall reveal whether the model is actually useful.